In [ ]:
from langchain_openai import ChatOpenAI
from typing import TypedDict, Annotated, List
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage

# ==========================================
# 1. 配置模型 (这是唯一变化的地方)
# ==========================================

# 建议：为了安全，尽量不要把 Key 硬编码在代码里
# 如果环境变量里没有 ARK_API_KEY，这里会提示你输入
# if not os.environ.get("ARK_API_KEY"):
#     os.environ["ARK_API_KEY"] = getpass.getpass("请输入你的火山引擎 API Key: ")

# 使用 ChatOpenAI 接入火山引擎
llm = ChatOpenAI(
    # 替换为官方示例中的 base_url
    base_url="https://ark.cn-beijing.volces.com/api/v3",
    # 替换为你的 API Key 环境变量名
    api_key="12a8708c-48cf-4825-9e14-4524d48d24e9",
    # 替换为官方示例中的 model ID (接入点 ID)
    model="doubao-seed-1-6-251015",
    temperature=0.7,
)

# 测试一下连接
print("正在测试 API 连接...")
try:
    test_res = llm.invoke("你好，你是谁？")
    print(f"✅ 连接成功！模型回复: {test_res.content}\n")
except Exception as e:
    print(f"❌ 连接失败: {e}")

# ==========================================
# 2. 定义图结构 (逻辑完全不用变)
# ==========================================

# 定义状态
class State(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

# 定义节点
def chatbot_node(state: State):
    messages = state["messages"]
    # 这里调用的 llm 已经是配置好的火山引擎模型了
    response = llm.invoke(messages)
    return {"messages": [response]}

# 构建图
workflow = StateGraph(State)
workflow.add_node("chatbot", chatbot_node)
workflow.add_edge(START, "chatbot")
workflow.add_edge("chatbot", END)
app = workflow.compile()

# ==========================================
# 3. 运行图
# ==========================================

print("--- 开始运行 LangGraph (API版) ---")
input_data = {"messages": [HumanMessage(content="用两句话介绍一下什么是 '豆包' 模型？")]}

for event in app.stream(input_data):
    for key, value in event.items():
        print(f"\n[节点: {key}] 完成")
        print(f"回复: {value['messages'][-1].content}")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 初始化模型 (复用你刚才配置成功的配置)
llm = ChatOpenAI(
    base_url="https://ark.cn-beijing.volces.com/api/v3",
    api_key="12a8708c-48cf-4825-9e14-4524d48d24e9",
    model="doubao-seed-1-6-251015",
    temperature=0.8, # 暴躁一点，调高温度
)

# 2. 定义模板
# system: 设定人设
# user: 留出 {code} 作为变量
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个资深且毒舌的代码审查员。你的任务是找出代码里的坏味道，并用讽刺但专业的语气指出来。不要给面子。"),
    ("user", "请审查这段代码：\n{code}")
])

# 3. 构建链 (LCEL语法: Prompt | Model | Parser)
# 这一行就是 LangChain 表达式语言 (LCEL)，非常优雅
chain = prompt | llm | StrOutputParser()

# 4. 运行
bad_code = """
def add(a, b):
    return a + b
"""

print("--- 正在暴躁 Review 中 ---")
# 这里的 input 字典会自动填入模板里的 {code}
response = chain.invoke({"code": bad_code})
print(response)

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Optional

# 1. 定义数据结构 (保持不变)
class UserInfo(BaseModel):
    name: Optional[str] = Field(description="用户的姓名")
    age: Optional[int] = Field(description="用户的年龄，如果没有提到则为null")
    intent: str = Field(description="用户的意图，例如：购买、咨询、投诉")
    urgency: int = Field(description="事情的紧急程度，1-10分")

# 2. 设置解析器 (这是核心变化)
# 解析器会自动生成一段提示语，告诉模型："请输出 JSON，Schema 如下..."
parser = PydanticOutputParser(pydantic_object=UserInfo)

# 3. 定义 Prompt
# 注意：必须把 {format_instructions} 放进去，这是解析器自动生成的"紧箍咒"
prompt = PromptTemplate(
    template="""你是一个信息提取助手。请从用户的输入中提取关键信息。
    
{format_instructions}

用户输入: {query}
""",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# 4. 初始化模型 (建议 temperature=0，让模型更严谨)
llm = ChatOpenAI(
    base_url="https://ark.cn-beijing.volces.com/api/v3",
    api_key="12a8708c-48cf-4825-9e14-4524d48d24e9",
    model="doubao-1-5-lite-32k-250115",
    temperature=0, # 关键：提取信息时，越低越好，不要创造性
)

# 5. 构建链
chain = prompt | llm | parser

# 6. 测试
user_input = "我是张三，今年28岁，我买的那个手机屏幕裂了，必须马上给我退货！气死我了！"

print("--- 正在强制提取 JSON ---")
try:
    data = chain.invoke({"query": user_input})
    print(f"解析成功！")
    print(f"姓名: {data.name}")
    print(f"意图: {data.intent}")
    print(f"紧急度: {data.urgency}")
    print(f"完整对象: {data}")
except Exception as e:
    print(f"解析失败: {e}")

--- 正在强制提取 JSON ---
解析成功！
姓名: 张三
意图: 退货
紧急度: 10
完整对象: name='张三' age=28 intent='退货' urgency=10
